# 🎮 Práctica 08: 3D Scatter Plot con Sprites de Pokémons

---

| Campo | Detalle |
|-------|--------|
| **Estudiante** | Francisco Garcia Garcia |
| **Matrícula** | 230758 |
| **Grupo** | 9°A - IDGS |
| **Materia** | Extracción de Conocimiento en Bases de Datos (ECBD) |
| **Fecha** | 06 de Agosto de 2026 |

---

## Objetivo

Construir un **Scatter Plot 3D interactivo** utilizando **Plotly** que permita visualizar y explorar las estadísticas base de los Pokémon (generaciones 1 a 9), diferenciando visualmente cada tipo principal mediante colores, integrando **sprites oficiales** en la información emergente, y aplicando filtros interactivos por generación, tipo y rango de estadísticas para identificar patrones, agrupaciones y valores atípicos.

## Fuentes de Datos

- **Dataset:** [lgreski/pokemonData](https://github.com/lgreski/pokemonData) — 1,215 Pokémon con estadísticas base (Gen 1–9), cortesía de pokemondb.net
- **Sprites:** [PokeAPI Sprites](https://github.com/PokeAPI/sprites) — Imágenes oficiales de cada Pokémon
- **Referencia:** [Unsupervised Learning: K-Means EDA (Kaggle)](https://www.kaggle.com/code/tanmay111999/unsupervised-learning-3-6-clusters-k-means-eda) — Técnicas de visualización 3D con Plotly

---

## 1. Importación de Librerías

Importamos las librerías necesarias para la manipulación, análisis y visualización de datos:
- **Pandas:** Manipulación y análisis de DataFrames
- **NumPy:** Operaciones numéricas y cálculos estadísticos
- **Plotly:** Visualizaciones 3D interactivas (graph_objects y express)
- **IPython.display:** Visualización enriquecida en Jupyter

In [1]:
# ============================================================
# Importación de Librerías
# ============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML, Image
import warnings

# Configuración general
warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_dark'
pio.renderers.default = 'notebook_connected'
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print('✅ Librerías importadas correctamente')
print(f'   📦 Pandas:  {pd.__version__}')
print(f'   📦 NumPy:   {np.__version__}')
print(f'   📦 Plotly:  {pio.__version__ if hasattr(pio, "__version__") else "disponible"}')

✅ Librerías importadas correctamente
   📦 Pandas:  2.2.2
   📦 NumPy:   2.3.1
   📦 Plotly:  disponible


---

## 2. Carga del Dataset

El dataset proviene del repositorio [lgreski/pokemonData](https://github.com/lgreski/pokemonData) en GitHub, que contiene estadísticas básicas de **1,025 Pokémon únicos** (con formas alternativas el total supera los 1,200 registros) de las **generaciones 1 a 9**, recopiladas de [pokemondb.net](https://pokemondb.net).

**Columnas del dataset:**
- `ID` — Número del Pokédex Nacional
- `Name` — Nombre del Pokémon
- `Form` — Forma/variante (Mega, Alolan, Galarian, etc.)
- `Type1`, `Type2` — Tipos principal y secundario
- `Total` — Suma total de estadísticas base
- `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed` — Estadísticas individuales
- `Generation` — Generación a la que pertenece

In [2]:
# ============================================================
# Carga del Dataset desde el repositorio de GitHub
# ============================================================
url_dataset = 'https://raw.githubusercontent.com/lgreski/pokemonData/master/Pokemon.csv'

# También se puede cargar localmente:
# df = pd.read_csv('Pokemon.csv')

df = pd.read_csv(url_dataset)

print(f'✅ Dataset cargado exitosamente')
print(f'   📊 Registros: {df.shape[0]}')
print(f'   📋 Columnas:  {df.shape[1]}')
print(f'   📁 Fuente:    lgreski/pokemonData (GitHub)')

✅ Dataset cargado exitosamente
   📊 Registros: 1215
   📋 Columnas:  13
   📁 Fuente:    lgreski/pokemonData (GitHub)


---

## 3. Inspección Inicial del Dataset

Realizamos una inspección completa del dataset utilizando las funciones estándar de Pandas para comprender su estructura, tipos de datos y distribución estadística.

### 3.1 Primeros registros (`head()`)

In [3]:
# ============================================================
# Primeros 10 registros del dataset
# ============================================================
df.head(10)

,ID,Name,Form,Type1,Type2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
0,1,Bulbasaur,,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,,Fire,,309,39,52,43,60,50,65,1
4,5,Charmeleon,,Fire,,405,58,64,58,80,65,80,1
5,6,Charizard,,Fire,Flying,534,78,84,78,109,85,100,1
6,7,Squirtle,,Water,,314,44,48,65,50,64,43,1
7,8,Wartortle,,Water,,405,59,63,80,65,80,58,1
8,9,Blastoise,,Water,,530,79,83,100,85,105,78,1
9,10,Caterpie,,Bug,,195,45,30,35,20,20,45,1


### 3.2 Dimensiones del Dataset (`shape`)

In [4]:
# ============================================================
# Dimensiones del dataset
# ============================================================
print(f'📐 Dimensiones del dataset:')
print(f'   Filas (registros):  {df.shape[0]}')
print(f'   Columnas (campos):  {df.shape[1]}')
print(f'\n📋 Nombres de columnas:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:2d}. {col}')

📐 Dimensiones del dataset:
   Filas (registros):  1215
   Columnas (campos):  13

📋 Nombres de columnas:
    1. ID
    2. Name
    3. Form
    4. Type1
    5. Type2
    6. Total
    7. HP
    8. Attack
    9. Defense
   10. Sp. Atk
   11. Sp. Def
   12. Speed
   13. Generation


### 3.3 Información del Dataset (`info()`)

In [5]:
# ============================================================
# Información detallada del dataset
# ============================================================
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          1215 non-null   int64 
 1   Name        1215 non-null   object
 2   Form        1215 non-null   object
 3   Type1       1215 non-null   object
 4   Type2       1215 non-null   object
 5   Total       1215 non-null   int64 
 6   HP          1215 non-null   int64 
 7   Attack      1215 non-null   int64 
 8   Defense     1215 non-null   int64 
 9   Sp. Atk     1215 non-null   int64 
 10  Sp. Def     1215 non-null   int64 
 11  Speed       1215 non-null   int64 
 12  Generation  1215 non-null   int64 
dtypes: int64(9), object(4)
memory usage: 123.5+ KB


### 3.4 Estadísticas Descriptivas (`describe()`)

In [6]:
# ============================================================
# Estadísticas descriptivas de columnas numéricas
# ============================================================
df.describe().round(2)

,ID,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
count,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00
mean,501.74,443.10,71.24,81.15,75.01,73.22,72.44,70.03,5.06
std,298.98,121.19,26.93,32.04,30.74,32.76,27.58,30.16,2.60
min,1.00,175.00,1.00,5.00,5.00,10.00,20.00,5.00,1.00
25%,240.50,332.00,52.00,57.00,52.00,50.00,51.00,45.00,3.00
50%,495.00,465.00,70.00,80.00,70.00,65.00,70.00,68.00,5.00
75%,753.50,521.00,85.00,100.00,91.00,95.00,90.00,91.00,7.00
max,1025.00,1125.00,255.00,190.00,250.00,194.00,250.00,200.00,9.00


In [7]:
# ============================================================
# Estadísticas descriptivas de columnas categóricas
# ============================================================
df.describe(include='object')

,Name,Form,Type1,Type2
count,1215,1215,1215,1215
unique,1026,207,18,19
top,Rotom,,Water,
freq,6,985,150,546


---

## 4. Limpieza y Normalización de Datos

Procedemos a limpiar y normalizar el dataset para garantizar la calidad de los datos antes del análisis:
1. Renombrar columnas a formato `snake_case` para consistencia
2. Limpiar espacios en blanco en valores categóricos
3. Identificar y tratar valores nulos
4. Detectar y eliminar registros duplicados

### 4.1 Estado ANTES de la Limpieza

In [8]:
# ============================================================
# Estado del dataset ANTES de la limpieza
# ============================================================
print('=' * 60)
print('📋 ESTADO ANTES DE LA LIMPIEZA')
print('=' * 60)

# Valores nulos
print('\n🔍 Valores nulos por columna:')
nulos_antes = df.isnull().sum()
print(nulos_antes[nulos_antes > 0] if nulos_antes.sum() > 0 else '   No se encontraron valores nulos explícitos')

# Valores en blanco o espacios en Type2
print(f'\n🔍 Valores en blanco/espacios en Type1: {(df["Type1"].str.strip() == "").sum()}')
print(f'🔍 Valores en blanco/espacios en Type2: {(df["Type2"].str.strip() == "").sum()}')

# Duplicados
print(f'\n🔍 Registros duplicados: {df.duplicated().sum()}')

# Columnas actuales
print(f'\n🔍 Nombres de columnas originales: {list(df.columns)}')

# Tipos únicos
print(f'\n🔍 Tipos únicos en Type1: {sorted(df["Type1"].str.strip().unique())}')

📋 ESTADO ANTES DE LA LIMPIEZA

🔍 Valores nulos por columna:
   No se encontraron valores nulos explícitos

🔍 Valores en blanco/espacios en Type1: 0
🔍 Valores en blanco/espacios en Type2: 546

🔍 Registros duplicados: 0

🔍 Nombres de columnas originales: ['ID', 'Name', 'Form', 'Type1', 'Type2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation']

🔍 Tipos únicos en Type1: ['Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water']


### 4.2 Proceso de Limpieza y Normalización

In [9]:
# ============================================================
# Proceso de limpieza y normalización
# ============================================================

# 1. Renombrar columnas a snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('.', '', regex=False)
)
print('✅ 1. Columnas renombradas a snake_case')
print(f'   {list(df.columns)}')

# 2. Limpiar espacios en blanco en columnas de texto
for col in ['name', 'form', 'type1', 'type2']:
    df[col] = df[col].str.strip()
print('\n✅ 2. Espacios en blanco eliminados de columnas de texto')

# 3. Reemplazar cadenas vacías en type2 por 'None' (Pokémon de un solo tipo)
df['type2'] = df['type2'].replace('', 'None')
print(f'\n✅ 3. Valores vacíos en type2 reemplazados por "None"')
print(f'   Pokémon con un solo tipo: {(df["type2"] == "None").sum()}')
print(f'   Pokémon con dos tipos:    {(df["type2"] != "None").sum()}')

# 4. Reemplazar cadenas vacías en form por 'Standard'
df['form'] = df['form'].replace('', 'Standard')
print(f'\n✅ 4. Valores vacíos en form reemplazados por "Standard"')

# 5. Verificar y eliminar duplicados
duplicados = df.duplicated().sum()
if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'\n✅ 5. Se eliminaron {duplicados} registros duplicados')
else:
    print(f'\n✅ 5. No se encontraron registros duplicados')

# 6. Verificar tipos de datos numéricos
cols_numericas = ['id', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(f'\n✅ 6. Tipos de datos numéricos verificados')

✅ 1. Columnas renombradas a snake_case
   ['id', 'name', 'form', 'type1', 'type2', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']

✅ 2. Espacios en blanco eliminados de columnas de texto

✅ 3. Valores vacíos en type2 reemplazados por "None"
   Pokémon con un solo tipo: 546
   Pokémon con dos tipos:    669

✅ 4. Valores vacíos en form reemplazados por "Standard"

✅ 5. No se encontraron registros duplicados

✅ 6. Tipos de datos numéricos verificados


### 4.3 Estado DESPUÉS de la Limpieza

In [10]:
# ============================================================
# Estado del dataset DESPUÉS de la limpieza
# ============================================================
print('=' * 60)
print('📋 ESTADO DESPUÉS DE LA LIMPIEZA')
print('=' * 60)

# Dimensiones
print(f'\n📐 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

# Valores nulos
nulos_despues = df.isnull().sum()
print(f'\n🔍 Valores nulos: {nulos_despues.sum()}')

# Duplicados
print(f'🔍 Registros duplicados: {df.duplicated().sum()}')

# Columnas normalizadas
print(f'\n📋 Columnas normalizadas: {list(df.columns)}')

# Tipos de datos
print(f'\n📊 Tipos de datos:')
for col in df.columns:
    print(f'   {col:20s} → {df[col].dtype}')

# Resumen de tipos de Pokémon
print(f'\n🎯 Tipos únicos de Pokémon (Type1): {df["type1"].nunique()}')
print(f'🎯 Generaciones: {sorted(df["generation"].unique())}')

# Muestra final
print(f'\n📄 Muestra del dataset limpio:')
df.head()

📋 ESTADO DESPUÉS DE LA LIMPIEZA

📐 Dimensiones: 1215 filas × 13 columnas

🔍 Valores nulos: 0
🔍 Registros duplicados: 0

📋 Columnas normalizadas: ['id', 'name', 'form', 'type1', 'type2', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']

📊 Tipos de datos:
   id                   → int64
   name                 → object
   form                 → object
   type1                → object
   type2                → object
   total                → int64
   hp                   → int64
   attack               → int64
   defense              → int64
   sp_atk               → int64
   sp_def               → int64
   speed                → int64
   generation           → int64

🎯 Tipos únicos de Pokémon (Type1): 18
🎯 Generaciones: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

📄 Muestra del dataset limpio:


,id,name,form,type1,type2,total,hp,attack,defense,sp_atk,sp_def,speed,generation
0,1,Bulbasaur,Standard,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,Standard,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,Standard,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,Standard,Fire,None,309,39,52,43,60,50,65,1
4,5,Charmeleon,Standard,Fire,None,405,58,64,58,80,65,80,1


### 4.4 Resumen Visual del Dataset Limpio

In [11]:
# ============================================================
# Distribución de Pokémon por Tipo Principal
# ============================================================
print('📊 Distribución de Pokémon por Tipo Principal:')
print('=' * 50)
tipo_counts = df['type1'].value_counts()
for tipo, count in tipo_counts.items():
    barra = '█' * (count // 5)
    print(f'   {tipo:12s} │ {barra} {count}')

print(f'\n📊 Distribución de Pokémon por Generación:')
print('=' * 50)
gen_counts = df['generation'].value_counts().sort_index()
for gen, count in gen_counts.items():
    barra = '█' * (count // 5)
    print(f'   Gen {int(gen):1d}      │ {barra} {count}')

📊 Distribución de Pokémon por Tipo Principal:
   Water        │ ██████████████████████████████ 150
   Normal       │ ██████████████████████████ 134
   Grass        │ ██████████████████████ 113
   Bug          │ ██████████████████ 91
   Psychic      │ ████████████████ 82
   Fire         │ ███████████████ 76
   Electric     │ ██████████████ 74
   Rock         │ █████████████ 68
   Dark         │ ███████████ 56
   Fighting     │ ██████████ 50
   Poison       │ █████████ 49
   Dragon       │ █████████ 49
   Ghost        │ █████████ 47
   Ground       │ █████████ 47
   Steel        │ █████████ 45
   Ice          │ ████████ 43
   Fairy        │ ██████ 31
   Flying       │ ██ 10

📊 Distribución de Pokémon por Generación:
   Gen 1      │ ██████████████████████████████ 151
   Gen 2      │ ████████████████████ 100
   Gen 3      │ ████████████████████████████ 141
   Gen 4      │ ███████████████████████ 118
   Gen 5      │ █████████████████████████████████ 165
   Gen 6      │ █████████████████████

---

## 5. Selección y Justificación de Variables Estadísticas

Para el análisis y la visualización 3D, seleccionamos las **6 estadísticas base** que definen las capacidades de combate de cada Pokémon:

| Variable | Descripción | Justificación |
|----------|-------------|---------------|
| `hp` | Puntos de Vida | Determina la resistencia total del Pokémon en combate |
| `attack` | Ataque Físico | Mide el poder de los movimientos físicos |
| `defense` | Defensa Física | Capacidad de resistir ataques físicos |
| `sp_atk` | Ataque Especial | Poder de los movimientos especiales (fuego, agua, etc.) |
| `sp_def` | Defensa Especial | Capacidad de resistir ataques especiales |
| `speed` | Velocidad | Determina el orden de turno en combate |

Estas 6 estadísticas son el estándar oficial de la franquicia Pokémon y permiten crear un **perfil estadístico completo** de cada Pokémon, identificando roles como tanques (alta defensa), sweepers (alto ataque/velocidad) o soportes (alta defensa especial/HP).

In [12]:
# ============================================================
# Selección de variables estadísticas para el análisis
# ============================================================
stats_cols = ['hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed']

print('📊 Variables estadísticas seleccionadas para el análisis:')
print('=' * 60)
for i, col in enumerate(stats_cols, 1):
    min_val = df[col].min()
    max_val = df[col].max()
    mean_val = df[col].mean()
    print(f'   {i}. {col:10s} │ Mín: {min_val:5.0f} │ Máx: {max_val:5.0f} │ Media: {mean_val:6.1f}')

print(f'\n📋 Total de variables seleccionadas: {len(stats_cols)}')
print(f'📋 Variable adicional disponible: "total" (suma de las 6 estadísticas)')

📊 Variables estadísticas seleccionadas para el análisis:
   1. hp         │ Mín:     1 │ Máx:   255 │ Media:   71.2
   2. attack     │ Mín:     5 │ Máx:   190 │ Media:   81.2
   3. defense    │ Mín:     5 │ Máx:   250 │ Media:   75.0
   4. sp_atk     │ Mín:    10 │ Máx:   194 │ Media:   73.2
   5. sp_def     │ Mín:    20 │ Máx:   250 │ Media:   72.4
   6. speed      │ Mín:     5 │ Máx:   200 │ Media:   70.0

📋 Total de variables seleccionadas: 6
📋 Variable adicional disponible: "total" (suma de las 6 estadísticas)


---

## 6. Creación de la Columna `promedio_estadisticas`

Calculamos una nueva columna que representa el **promedio aritmético** de las 6 estadísticas base. Esta métrica unificada permite comparar Pokémon de forma directa y será utilizada como uno de los ejes del gráfico 3D.

In [13]:
# ============================================================
# Creación de la columna promedio_estadisticas
# ============================================================
df['promedio_estadisticas'] = df[stats_cols].mean(axis=1).round(2)

print('✅ Columna "promedio_estadisticas" creada exitosamente')
print(f'\n📊 Estadísticas del promedio:')
print(f'   Mínimo:  {df["promedio_estadisticas"].min():.2f}')
print(f'   Máximo:  {df["promedio_estadisticas"].max():.2f}')
print(f'   Media:   {df["promedio_estadisticas"].mean():.2f}')
print(f'   Mediana: {df["promedio_estadisticas"].median():.2f}')
print(f'   Std:     {df["promedio_estadisticas"].std():.2f}')

# Top 10 Pokémon con mayor promedio
print(f'\n🏆 Top 10 Pokémon con mayor promedio de estadísticas:')
print('=' * 60)
top10 = df.nlargest(10, 'promedio_estadisticas')[['id', 'name', 'form', 'type1', 'generation', 'promedio_estadisticas']]
for idx, row in top10.iterrows():
    print(f'   #{row["id"]:4.0f} {row["name"]:20s} ({row["form"]:15s}) │ {row["type1"]:10s} │ Gen {row["generation"]:.0f} │ Prom: {row["promedio_estadisticas"]:.2f}')

# Bottom 5
print(f'\n📉 Top 5 Pokémon con menor promedio de estadísticas:')
bottom5 = df.nsmallest(5, 'promedio_estadisticas')[['id', 'name', 'type1', 'generation', 'promedio_estadisticas']]
for idx, row in bottom5.iterrows():
    print(f'   #{row["id"]:4.0f} {row["name"]:20s} │ {row["type1"]:10s} │ Gen {row["generation"]:.0f} │ Prom: {row["promedio_estadisticas"]:.2f}')

✅ Columna "promedio_estadisticas" creada exitosamente

📊 Estadísticas del promedio:
   Mínimo:  29.17
   Máximo:  187.50
   Media:   73.85
   Mediana: 77.50
   Std:     20.20

🏆 Top 10 Pokémon con mayor promedio de estadísticas:
   # 890 Eternatus            (Eternamax      ) │ Poison     │ Gen 8 │ Prom: 187.50
   # 150 Mewtwo               (Mega Mewtwo X  ) │ Psychic    │ Gen 6 │ Prom: 130.00
   # 150 Mewtwo               (Mega Mewtwo Y  ) │ Psychic    │ Gen 6 │ Prom: 130.00
   # 384 Rayquaza             (Mega Rayquaza  ) │ Dragon     │ Gen 6 │ Prom: 130.00
   # 382 Kyogre               (Primal Kyogre  ) │ Water      │ Gen 6 │ Prom: 128.33
   # 383 Groudon              (Primal Groudon ) │ Ground     │ Gen 6 │ Prom: 128.33
   # 800 Necrozma             (Ultra Necrozma ) │ Psychic    │ Gen 7 │ Prom: 125.67
   # 493 Arceus               (Standard       ) │ Normal     │ Gen 4 │ Prom: 120.00
   # 718 Zygarde              (Complete Forme ) │ Dragon     │ Gen 7 │ Prom: 118.00
   # 646 Kyurem

---

## 7. Análisis Estadístico Descriptivo

Realizamos un análisis estadístico completo que incluye **media, mediana, mínimo, máximo y desviación estándar** de las estadísticas seleccionadas, segmentado por tipo principal y por generación.

### 7.1 Estadísticas Descriptivas Generales

In [14]:
# ============================================================
# Estadísticas descriptivas generales
# ============================================================
stats_resumen = df[stats_cols + ['promedio_estadisticas']].agg(
    ['mean', 'median', 'min', 'max', 'std']
).round(2)

stats_resumen.index = ['Media', 'Mediana', 'Mínimo', 'Máximo', 'Desv. Estándar']

print('📊 Estadísticas Descriptivas Generales:')
print('=' * 80)
display(stats_resumen)

📊 Estadísticas Descriptivas Generales:


,hp,attack,defense,sp_atk,sp_def,speed,promedio_estadisticas
Media,71.24,81.15,75.01,73.22,72.44,70.03,73.85
Mediana,70.00,80.00,70.00,65.00,70.00,68.00,77.50
Mínimo,1.00,5.00,5.00,10.00,20.00,5.00,29.17
Máximo,255.00,190.00,250.00,194.00,250.00,200.00,187.50
Desv. Estándar,26.93,32.04,30.74,32.76,27.58,30.16,20.20


### 7.2 Estadísticas por Tipo Principal

In [15]:
# ============================================================
# Promedio de estadísticas por Tipo Principal
# ============================================================
stats_por_tipo = df.groupby('type1')[stats_cols + ['promedio_estadisticas']].mean().round(2)
stats_por_tipo = stats_por_tipo.sort_values('promedio_estadisticas', ascending=False)

print('📊 Promedio de Estadísticas por Tipo Principal (ordenado por promedio):')
print('=' * 80)
display(stats_por_tipo)

📊 Promedio de Estadísticas por Tipo Principal (ordenado por promedio):


,hp,attack,defense,sp_atk,sp_def,speed,promedio_estadisticas
type1,,,,,,,
Dragon,84.57,103.82,80.82,90.12,83.57,84.65,87.93
Steel,71.18,92.51,114.27,75.16,79.22,57.56,81.65
Psychic,73.84,75.65,71.52,98.72,86.30,80.37,81.07
Fighting,75.74,104.96,76.44,55.16,69.66,76.12,76.35
Fire,70.76,84.47,69.32,86.83,71.64,73.96,76.16
Flying,70.90,81.90,67.40,72.60,70.90,86.80,75.08
Dark,72.89,85.48,70.98,72.27,70.55,77.54,74.95
Electric,63.84,73.15,65.84,88.68,70.41,87.50,74.90
Fairy,72.13,71.06,73.65,78.13,87.19,67.06,74.87


### 7.3 Estadísticas por Generación

In [16]:
# ============================================================
# Promedio de estadísticas por Generación
# ============================================================
stats_por_gen = df.groupby('generation')[stats_cols + ['promedio_estadisticas']].agg(
    ['mean', 'median', 'std']
).round(2)

# Simplificar para mostrar solo el promedio_estadisticas por generación
resumen_gen = df.groupby('generation').agg(
    total_pokemon=('id', 'count'),
    prom_hp=('hp', 'mean'),
    prom_attack=('attack', 'mean'),
    prom_defense=('defense', 'mean'),
    prom_sp_atk=('sp_atk', 'mean'),
    prom_sp_def=('sp_def', 'mean'),
    prom_speed=('speed', 'mean'),
    prom_general=('promedio_estadisticas', 'mean'),
    std_general=('promedio_estadisticas', 'std')
).round(2)

print('📊 Resumen Estadístico por Generación:')
print('=' * 100)
display(resumen_gen)

📊 Resumen Estadístico por Generación:


,total_pokemon,prom_hp,prom_attack,prom_defense,prom_sp_atk,prom_sp_def,prom_speed,prom_general,std_general
generation,,,,,,,,,
1,151,64.21,72.91,68.23,67.14,66.09,69.07,67.94,16.65
2,100,70.98,68.26,69.69,64.50,72.34,61.41,67.86,18.74
3,141,65.43,73.94,69.48,68.91,67.04,63.46,68.04,19.43
4,118,72.22,79.13,76.58,74.51,75.75,69.70,74.65,19.87
5,165,71.71,82.44,72.08,71.99,68.31,68.37,72.48,17.99
6,131,72.55,94.92,87.88,89.27,84.36,76.63,84.27,22.61
7,122,70.43,86.51,78.28,74.57,73.82,69.28,75.48,20.36
8,147,75.35,86.20,76.83,75.01,73.90,73.51,76.80,21.45
9,140,78.69,83.85,77.01,72.67,73.00,76.94,77.03,19.31


---

## 8. Preparación de Variables para la Visualización

Preparamos y ordenamos las variables de **generación** y **tipo principal** para su correcta representación en la visualización 3D. Asignamos una paleta de **colores oficiales** de los tipos Pokémon.

In [17]:
# ============================================================
# Preparación de variables de generación y tipo
# ============================================================

# 1. Asegurar que generation sea entero y esté ordenada
df['generation'] = df['generation'].astype(int)
generaciones_ordenadas = sorted(df['generation'].unique())
print(f'✅ Generaciones disponibles (ordenadas): {generaciones_ordenadas}')

# 2. Ordenar tipos principales y asignar código numérico para eje Y
tipos_ordenados = sorted(df['type1'].unique())
tipo_a_numero = {tipo: i for i, tipo in enumerate(tipos_ordenados)}
df['type1_num'] = df['type1'].map(tipo_a_numero)

print(f'\n✅ Tipos principales ({len(tipos_ordenados)} tipos):')
for tipo, num in tipo_a_numero.items():
    count = (df['type1'] == tipo).sum()
    print(f'   {num:2d}. {tipo:12s} → {count} Pokémon')

# 3. Paleta de colores oficial de tipos Pokémon
colores_tipo = {
    'Normal': '#A8A878',
    'Fire': '#F08030',
    'Water': '#6890F0',
    'Electric': '#F8D030',
    'Grass': '#78C850',
    'Ice': '#98D8D8',
    'Fighting': '#C03028',
    'Poison': '#A040A0',
    'Ground': '#E0C068',
    'Flying': '#A890F0',
    'Psychic': '#F85888',
    'Bug': '#A8B820',
    'Rock': '#B8A038',
    'Ghost': '#705898',
    'Dragon': '#7038F8',
    'Dark': '#705848',
    'Steel': '#B8B8D0',
    'Fairy': '#EE99AC'
}

# Asignar color a cada Pokémon según su tipo
df['color_tipo'] = df['type1'].map(colores_tipo)

print(f'\n✅ Paleta de colores asignada ({len(colores_tipo)} tipos con color)')
print(f'   Pokémon sin color asignado: {df["color_tipo"].isna().sum()}')

✅ Generaciones disponibles (ordenadas): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

✅ Tipos principales (18 tipos):
    0. Bug          → 91 Pokémon
    1. Dark         → 56 Pokémon
    2. Dragon       → 49 Pokémon
    3. Electric     → 74 Pokémon
    4. Fairy        → 31 Pokémon
    5. Fighting     → 50 Pokémon
    6. Fire         → 76 Pokémon
    7. Flying       → 10 Pokémon
    8. Ghost        → 47 Pokémon
    9. Grass        → 113 Pokémon
   10. Ground       → 47 Pokémon
   11. Ice          → 43 Pokémon
   12. Normal       → 134 Pokémon
   13. Poison       → 49 Pokémon
   14. Psychic      → 82 Pokémon
   15. Rock         → 68 Pokémon
   16. Steel        → 45 Pokémon
   17. Water        → 150 Pokémon

✅ Paleta de colores asignada (18 tipos con color)
   Pokémon sin color asignado: 0


---

## 9. Obtención y Validación de Sprites de Pokémon

Los sprites (imágenes pequeñas) de cada Pokémon se obtienen del repositorio de [PokeAPI Sprites](https://github.com/PokeAPI/sprites) en GitHub. La URL de cada sprite se construye a partir del **ID del Pokédex Nacional**:

```
https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{id}.png
```

Esto nos permite integrar las imágenes directamente en la visualización interactiva sin necesidad de descargarlas localmente.

In [18]:
# ============================================================
# Construcción de URLs de sprites para cada Pokémon
# ============================================================
base_url_sprite = 'https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/'

# Construir la URL del sprite basada en el ID del Pokédex
df['sprite_url'] = df['id'].astype(int).apply(lambda x: f'{base_url_sprite}{x}.png')

print('✅ URLs de sprites construidas para cada Pokémon')
print(f'   Total de URLs generadas: {len(df)}')
print(f'\n📋 Ejemplos de URLs de sprites:')
for _, row in df.head(5).iterrows():
    print(f'   #{row["id"]:4.0f} {row["name"]:15s} → {row["sprite_url"]}')

✅ URLs de sprites construidas para cada Pokémon
   Total de URLs generadas: 1215

📋 Ejemplos de URLs de sprites:
   #   1 Bulbasaur       → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/1.png
   #   2 Ivysaur         → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/2.png
   #   3 Venusaur        → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/3.png
   #   4 Charmander      → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/4.png
   #   5 Charmeleon      → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/5.png


In [19]:
# ============================================================
# Validación de sprites con una muestra representativa
# ============================================================
import urllib.request

# Verificar sprites de una muestra (1 por generación)
print('🔍 Verificación de sprites (muestra por generación):')
print('=' * 60)

muestra = df.drop_duplicates(subset='generation').sort_values('generation')
sprites_validos = 0
sprites_totales = 0

for _, row in muestra.iterrows():
    sprites_totales += 1
    try:
        req = urllib.request.Request(row['sprite_url'], method='HEAD')
        response = urllib.request.urlopen(req, timeout=5)
        status = response.getcode()
        if status == 200:
            sprites_validos += 1
            print(f'   ✅ Gen {row["generation"]} │ #{row["id"]:4.0f} {row["name"]:15s} │ Sprite disponible')
        else:
            print(f'   ⚠️ Gen {row["generation"]} │ #{row["id"]:4.0f} {row["name"]:15s} │ Status: {status}')
    except Exception as e:
        print(f'   ❌ Gen {row["generation"]} │ #{row["id"]:4.0f} {row["name"]:15s} │ Error: {str(e)[:40]}')

print(f'\n📊 Resultado: {sprites_validos}/{sprites_totales} sprites verificados correctamente')

🔍 Verificación de sprites (muestra por generación):


   ✅ Gen 1 │ #   1 Bulbasaur       │ Sprite disponible


   ✅ Gen 2 │ # 152 Chikorita       │ Sprite disponible
   ✅ Gen 3 │ # 252 Treecko         │ Sprite disponible


   ✅ Gen 4 │ # 387 Turtwig         │ Sprite disponible


   ✅ Gen 5 │ # 494 Victini         │ Sprite disponible
   ✅ Gen 6 │ #   3 Venusaur        │ Sprite disponible


   ✅ Gen 7 │ #  19 Rattata         │ Sprite disponible


   ✅ Gen 8 │ #  52 Meowth          │ Sprite disponible


   ✅ Gen 9 │ # 128 Tauros          │ Sprite disponible

📊 Resultado: 9/9 sprites verificados correctamente


In [20]:
# ============================================================
# Vista previa de sprites (primeros 5 Pokémon)
# ============================================================
print('🖼️ Vista previa de sprites (Pokémon iniciales):')
print('=' * 60)

html_preview = '<div style="display: flex; gap: 20px; align-items: center; flex-wrap: wrap; background: #1a1a2e; padding: 20px; border-radius: 10px;">'
for _, row in df.head(9).iterrows():
    html_preview += f'''
    <div style="text-align: center; color: white;">
        <img src="{row['sprite_url']}" width="80" height="80" style="image-rendering: pixelated;">
        <br><span style="font-size: 11px;">#{int(row['id'])} {row['name']}</span>
        <br><span style="font-size: 10px; color: {row['color_tipo']};">{row['type1']}</span>
    </div>'''
html_preview += '</div>'

display(HTML(html_preview))

🖼️ Vista previa de sprites (Pokémon iniciales):


---

### ✅ Resumen del Dataset Preparado

El dataset está listo para la visualización 3D con las siguientes columnas adicionales:
- `promedio_estadisticas` — Promedio de las 6 estadísticas base
- `type1_num` — Codificación numérica del tipo principal
- `color_tipo` — Color hexadecimal asociado al tipo
- `sprite_url` — URL del sprite oficial del Pokémon

In [21]:
# ============================================================
# Resumen final del dataset preparado
# ============================================================
print('📋 RESUMEN DEL DATASET PREPARADO PARA VISUALIZACIÓN 3D')
print('=' * 60)
print(f'   📊 Total de registros:    {len(df)}')
print(f'   📋 Total de columnas:     {len(df.columns)}')
print(f'   🎮 Generaciones:          {df["generation"].nunique()} ({df["generation"].min()}-{df["generation"].max()})')
print(f'   🎯 Tipos principales:     {df["type1"].nunique()}')
print(f'   📈 Rango de promedios:    {df["promedio_estadisticas"].min():.1f} - {df["promedio_estadisticas"].max():.1f}')
print(f'   🖼️ Sprites configurados:  {df["sprite_url"].notna().sum()}')
print(f'\n📋 Columnas del dataset final:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:2d}. {col}')

df.head(3)

📋 RESUMEN DEL DATASET PREPARADO PARA VISUALIZACIÓN 3D
   📊 Total de registros:    1215
   📋 Total de columnas:     17
   🎮 Generaciones:          9 (1-9)
   🎯 Tipos principales:     18
   📈 Rango de promedios:    29.2 - 187.5
   🖼️ Sprites configurados:  1215

📋 Columnas del dataset final:
    1. id
    2. name
    3. form
    4. type1
    5. type2
    6. total
    7. hp
    8. attack
    9. defense
   10. sp_atk
   11. sp_def
   12. speed
   13. generation
   14. promedio_estadisticas
   15. type1_num
   16. color_tipo
   17. sprite_url


,id,name,form,type1,type2,total,hp,attack,defense,sp_atk,sp_def,speed,generation,promedio_estadisticas,type1_num,color_tipo,sprite_url
0,1,Bulbasaur,Standard,Grass,Poison,318,45,49,49,65,65,45,1,53.0,9,#78C850,https://raw.githubusercontent.com/PokeAPI/spri...
1,2,Ivysaur,Standard,Grass,Poison,405,60,62,63,80,80,60,1,67.5,9,#78C850,https://raw.githubusercontent.com/PokeAPI/spri...
2,3,Venusaur,Standard,Grass,Poison,525,80,82,83,100,100,80,1,87.5,9,#78C850,https://raw.githubusercontent.com/PokeAPI/spri...


---

## 10. Construcción del Scatter Plot 3D Interactivo

Construimos la primera versión del gráfico 3D interactivo utilizando `plotly.graph_objects.Scatter3d` con los siguientes ejes:
- **Eje X:** Generación (1–9)
- **Eje Y:** Tipo principal (codificado numéricamente)
- **Eje Z:** Promedio de estadísticas base

Cada punto representa un Pokémon, diferenciado por color según su tipo principal.

In [22]:
# ============================================================
# Scatter Plot 3D Base - Primera versión
# ============================================================

fig = go.Figure()

# Agregar una traza por cada tipo de Pokémon para la leyenda
for tipo in sorted(df['type1'].unique()):
    df_tipo = df[df['type1'] == tipo]
    color = colores_tipo.get(tipo, '#FFFFFF')
    
    fig.add_trace(go.Scatter3d(
        x=df_tipo['generation'],
        y=df_tipo['type1_num'],
        z=df_tipo['promedio_estadisticas'],
        mode='markers',
        name=tipo,
        marker=dict(
            size=5,
            color=color,
            opacity=0.8,
            line=dict(width=0.5, color='white')
        ),
        text=df_tipo['name'],
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Tipo: ' + tipo + '<br>'
            'Generación: %{x}<br>'
            'Promedio Stats: %{z:.1f}<br>'
            '<extra></extra>'
        )
    ))

fig.update_layout(
    title='Scatter Plot 3D - Pokémon por Generación, Tipo y Estadísticas',
    scene=dict(
        xaxis_title='Generación',
        yaxis_title='Tipo Principal',
        zaxis_title='Promedio Estadísticas'
    ),
    width=900,
    height=700,
    template='plotly_dark'
)

fig.show()
print('✅ Scatter Plot 3D base construido exitosamente')

✅ Scatter Plot 3D base construido exitosamente


---

## 11. Diferenciación Visual por Tipo y Hover Detallado

Mejoramos la visualización con:
- **Colores oficiales** por tipo Pokémon
- **Tamaño de marcadores** proporcional al promedio de estadísticas
- **Hover enriquecido** mostrando nombre, tipo, generación, promedio y todas las estadísticas individuales

In [23]:
# ============================================================
# Scatter Plot 3D con hover detallado y marcadores proporcionales
# ============================================================

fig2 = go.Figure()

for tipo in sorted(df['type1'].unique()):
    df_tipo = df[df['type1'] == tipo]
    color = colores_tipo.get(tipo, '#FFFFFF')
    
    # Tamaño proporcional al promedio de estadísticas (normalizado)
    sizes = 3 + (df_tipo['promedio_estadisticas'] - df['promedio_estadisticas'].min()) / \
            (df['promedio_estadisticas'].max() - df['promedio_estadisticas'].min()) * 12
    
    # Construir texto de hover completo
    hover_texts = []
    for _, row in df_tipo.iterrows():
        hover_text = (
            f"<b>{row['name']}</b>"
            f"<br>━━━━━━━━━━━━━━━━━━━━"
            f"<br>🎯 Tipo: {row['type1']}"
            f"{'/' + row['type2'] if row['type2'] != 'None' else ''}"
            f"<br>🎮 Generación: {int(row['generation'])}"
            f"<br>📊 Promedio: {row['promedio_estadisticas']:.1f}"
            f"<br>━━━━━━━━━━━━━━━━━━━━"
            f"<br>❤️ HP: {int(row['hp'])}"
            f"<br>⚔️ Attack: {int(row['attack'])}"
            f"<br>🛡️ Defense: {int(row['defense'])}"
            f"<br>🔮 Sp.Atk: {int(row['sp_atk'])}"
            f"<br>🔰 Sp.Def: {int(row['sp_def'])}"
            f"<br>💨 Speed: {int(row['speed'])}"
        )
        hover_texts.append(hover_text)
    
    fig2.add_trace(go.Scatter3d(
        x=df_tipo['generation'],
        y=df_tipo['type1_num'],
        z=df_tipo['promedio_estadisticas'],
        mode='markers',
        name=tipo,
        marker=dict(
            size=sizes,
            color=color,
            opacity=0.85,
            line=dict(width=0.5, color='rgba(255,255,255,0.3)'),
            symbol='circle'
        ),
        text=hover_texts,
        hoverinfo='text',
        hovertemplate='%{text}<extra></extra>'
    ))

# Configurar etiquetas del eje Y con nombres de tipos
tipo_tickvals = list(tipo_a_numero.values())
tipo_ticktext = list(tipo_a_numero.keys())

fig2.update_layout(
    title=dict(
        text='🎮 Scatter Plot 3D - Pokémon por Generación, Tipo y Promedio de Estadísticas',
        font=dict(size=16)
    ),
    scene=dict(
        xaxis=dict(title='Generación', tickvals=generaciones_ordenadas, dtick=1),
        yaxis=dict(title='Tipo Principal', tickvals=tipo_tickvals, ticktext=tipo_ticktext, tickfont=dict(size=9)),
        zaxis=dict(title='Promedio Estadísticas Base'),
        camera=dict(eye=dict(x=1.8, y=1.8, z=0.8))
    ),
    width=1000,
    height=750,
    template='plotly_dark',
    legend=dict(
        title='Tipo Pokémon',
        font=dict(size=10),
        itemsizing='constant'
    )
)

fig2.show()
print('✅ Scatter Plot 3D con hover detallado construido exitosamente')

✅ Scatter Plot 3D con hover detallado construido exitosamente


---

## 12. Integración de Sprites de Pokémon

Integramos los **sprites oficiales** de los Pokémon en la visualización. Dado que Plotly 3D no soporta imágenes directamente en tooltips, creamos un **panel HTML interactivo** complementario que muestra el sprite y las estadísticas al seleccionar un Pokémon.

Además, construimos una galería interactiva organizada por generación donde se pueden visualizar todos los sprites con su información.

In [24]:
# ============================================================
# Galería interactiva de sprites por generación
# ============================================================

def crear_galeria_sprites(df_input, generacion, max_pokemon=30):
    """Crea una galería HTML de sprites para una generación específica."""
    df_gen = df_input[df_input['generation'] == generacion].head(max_pokemon)
    
    html = f'''
    <div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); 
                padding: 20px; border-radius: 15px; margin: 10px 0;">
        <h3 style="color: #FFD700; text-align: center; margin-bottom: 15px;">🎮 Generación {generacion} — {len(df_gen)} Pokémon</h3>
        <div style="display: flex; flex-wrap: wrap; gap: 10px; justify-content: center;">'''
    
    for _, row in df_gen.iterrows():
        color = row['color_tipo']
        html += f'''
        <div style="background: rgba(255,255,255,0.08); border-radius: 10px; padding: 8px; 
                    width: 100px; text-align: center; border: 1px solid {color}40;
                    transition: transform 0.2s;">
            <img src="{row['sprite_url']}" width="64" height="64" 
                 style="image-rendering: pixelated;" loading="lazy">
            <div style="color: white; font-size: 10px; font-weight: bold;">#{int(row['id'])} {row['name']}</div>
            <div style="color: {color}; font-size: 9px;">{row['type1']}</div>
            <div style="color: #aaa; font-size: 8px;">Prom: {row['promedio_estadisticas']:.1f}</div>
        </div>'''
    
    html += '</div></div>'
    return html

# Mostrar galería para las primeras 3 generaciones
for gen in [1, 2, 3]:
    display(HTML(crear_galeria_sprites(df, gen)))

In [25]:
# ============================================================
# Galería de sprites - Generaciones 4 a 6
# ============================================================
for gen in [4, 5, 6]:
    display(HTML(crear_galeria_sprites(df, gen)))

In [26]:
# ============================================================
# Galería de sprites - Generaciones 7 a 9
# ============================================================
for gen in [7, 8, 9]:
    display(HTML(crear_galeria_sprites(df, gen)))

---

## 13. Scatter Plot 3D con Filtros Interactivos

Agregamos **filtros interactivos** al gráfico 3D utilizando los `updatemenus` de Plotly:
- **Filtro por generación:** Botones dropdown para mostrar una generación específica o todas
- **Filtro por tipo:** Utilizando la leyenda interactiva (click para ocultar/mostrar)
- **Controles de visualización:** Cambiar el tamaño, opacidad y ángulo de cámara

In [27]:
# ============================================================
# Scatter Plot 3D con Filtros Interactivos por Generación
# ============================================================

fig3 = go.Figure()

# Crear trazas organizadas por tipo
tipos_unicos = sorted(df['type1'].unique())

for tipo in tipos_unicos:
    df_tipo = df[df['type1'] == tipo]
    color = colores_tipo.get(tipo, '#FFFFFF')
    
    sizes = 3 + (df_tipo['promedio_estadisticas'] - df['promedio_estadisticas'].min()) / \
            (df['promedio_estadisticas'].max() - df['promedio_estadisticas'].min()) * 12
    
    hover_texts = []
    for _, row in df_tipo.iterrows():
        hover_text = (
            f"<b>#{int(row['id'])} {row['name']}</b>"
            f"<br>━━━━━━━━━━━━━━━━"
            f"<br>🎯 Tipo: {row['type1']}"
            f"{'/' + row['type2'] if row['type2'] != 'None' else ''}"
            f"<br>🎮 Gen: {int(row['generation'])}"
            f"<br>📊 Promedio: {row['promedio_estadisticas']:.1f}"
            f"<br>━━━━━━━━━━━━━━━━"
            f"<br>❤️ HP: {int(row['hp'])} | ⚔️ Atk: {int(row['attack'])}"
            f"<br>🛡️ Def: {int(row['defense'])} | 🔮 SpA: {int(row['sp_atk'])}"
            f"<br>🔰 SpD: {int(row['sp_def'])} | 💨 Spd: {int(row['speed'])}"
        )
        hover_texts.append(hover_text)
    
    fig3.add_trace(go.Scatter3d(
        x=df_tipo['generation'],
        y=df_tipo['type1_num'],
        z=df_tipo['promedio_estadisticas'],
        mode='markers',
        name=tipo,
        marker=dict(
            size=sizes,
            color=color,
            opacity=0.85,
            line=dict(width=0.5, color='rgba(255,255,255,0.3)'),
        ),
        text=hover_texts,
        hoverinfo='text',
        hovertemplate='%{text}<extra></extra>',
        customdata=df_tipo[['generation']].values
    ))

# Crear botones de filtro por generación
buttons_gen = []

# Botón "Todas las generaciones"
buttons_gen.append(dict(
    label='Todas',
    method='update',
    args=[{'visible': [True] * len(tipos_unicos)}]
))

# Botones por cada generación
for gen in generaciones_ordenadas:
    visibility = []
    for tipo in tipos_unicos:
        df_tipo_gen = df[(df['type1'] == tipo) & (df['generation'] == gen)]
        visibility.append(len(df_tipo_gen) > 0)
    buttons_gen.append(dict(
        label=f'Gen {gen}',
        method='update',
        args=[{'visible': visibility}]
    ))

# Crear botones de ángulo de cámara
buttons_camera = [
    dict(label='Vista General', method='relayout',
         args=[{'scene.camera.eye': dict(x=1.8, y=1.8, z=0.8)}]),
    dict(label='Vista Frontal', method='relayout',
         args=[{'scene.camera.eye': dict(x=0.1, y=2.5, z=0.5)}]),
    dict(label='Vista Superior', method='relayout',
         args=[{'scene.camera.eye': dict(x=0.1, y=0.1, z=2.5)}]),
    dict(label='Vista Lateral', method='relayout',
         args=[{'scene.camera.eye': dict(x=2.5, y=0.1, z=0.5)}]),
]

fig3.update_layout(
    title=dict(
        text='🎮 3D Scatter Plot Interactivo — Pokémon (Gen 1–9)',
        font=dict(size=18, color='#FFD700')
    ),
    scene=dict(
        xaxis=dict(
            title='Generación',
            tickvals=generaciones_ordenadas,
            dtick=1,
            backgroundcolor='rgba(0,0,0,0)',
            gridcolor='rgba(255,255,255,0.1)'
        ),
        yaxis=dict(
            title='Tipo Principal',
            tickvals=tipo_tickvals,
            ticktext=tipo_ticktext,
            tickfont=dict(size=8),
            backgroundcolor='rgba(0,0,0,0)',
            gridcolor='rgba(255,255,255,0.1)'
        ),
        zaxis=dict(
            title='Promedio Estadísticas',
            backgroundcolor='rgba(0,0,0,0)',
            gridcolor='rgba(255,255,255,0.1)'
        ),
        camera=dict(eye=dict(x=1.8, y=1.8, z=0.8)),
        bgcolor='rgba(15,12,41,0.95)'
    ),
    updatemenus=[
        dict(
            buttons=buttons_gen,
            direction='down',
            showactive=True,
            x=0.02,
            xanchor='left',
            y=1.05,
            yanchor='top',
            bgcolor='rgba(50,50,80,0.8)',
            font=dict(color='white', size=10),
            bordercolor='rgba(255,215,0,0.5)'
        ),
        dict(
            buttons=buttons_camera,
            direction='down',
            showactive=True,
            x=0.15,
            xanchor='left',
            y=1.05,
            yanchor='top',
            bgcolor='rgba(50,50,80,0.8)',
            font=dict(color='white', size=10),
            bordercolor='rgba(255,215,0,0.5)'
        )
    ],
    annotations=[
        dict(text='Generación:', showarrow=False,
             x=0.02, y=1.09, xref='paper', yref='paper',
             font=dict(size=11, color='#FFD700')),
        dict(text='Cámara:', showarrow=False,
             x=0.15, y=1.09, xref='paper', yref='paper',
             font=dict(size=11, color='#FFD700'))
    ],
    width=1100,
    height=800,
    template='plotly_dark',
    legend=dict(
        title=dict(text='🎯 Tipo Pokémon', font=dict(size=12, color='#FFD700')),
        font=dict(size=10),
        itemsizing='constant',
        bgcolor='rgba(0,0,0,0.5)',
        bordercolor='rgba(255,215,0,0.3)',
        borderwidth=1
    ),
    paper_bgcolor='rgba(15,12,41,1)',
    plot_bgcolor='rgba(15,12,41,1)'
)

fig3.show()
print('✅ Scatter Plot 3D con filtros interactivos construido exitosamente')
print('   📌 Usa los dropdown de "Generación" y "Cámara" para filtrar y cambiar la vista')
print('   📌 Haz click en los tipos de la leyenda para ocultar/mostrar')
print('   📌 Pasa el cursor sobre los puntos para ver detalles completos')

✅ Scatter Plot 3D con filtros interactivos construido exitosamente
   📌 Usa los dropdown de "Generación" y "Cámara" para filtrar y cambiar la vista
   📌 Haz click en los tipos de la leyenda para ocultar/mostrar
   📌 Pasa el cursor sobre los puntos para ver detalles completos


---

## 14. Panel Interactivo de Sprites con Estadísticas

Creamos un panel complementario que combina los sprites con un resumen visual de estadísticas, mostrando los **Top Pokémon por tipo** con sus sprites oficiales y barras de estadísticas.

In [28]:
# ============================================================
# Panel de Sprites - Top Pokémon por Tipo con Barras de Stats
# ============================================================

def crear_panel_top_por_tipo(df_input, top_n=3):
    """Crea un panel HTML mostrando los top Pokémon por cada tipo."""
    html = '''
    <div style="background: linear-gradient(180deg, #0f0c29, #1a1a3e); 
                padding: 25px; border-radius: 15px; margin: 10px 0;">
        <h2 style="color: #FFD700; text-align: center; margin-bottom: 20px;">
            🏆 Top 3 Pokémon por Tipo — Basado en Promedio de Estadísticas
        </h2>
        <div style="display: flex; flex-wrap: wrap; gap: 15px; justify-content: center;">'''
    
    for tipo in sorted(df_input['type1'].unique()):
        color = colores_tipo.get(tipo, '#FFFFFF')
        top_pokemon = df_input[df_input['type1'] == tipo].nlargest(top_n, 'promedio_estadisticas')
        
        html += f'''
        <div style="background: rgba(255,255,255,0.05); border-radius: 12px; padding: 12px;
                    width: 260px; border: 1px solid {color}50;">
            <h4 style="color: {color}; text-align: center; margin: 0 0 10px 0; font-size: 14px;">
                {tipo}
            </h4>
            <div style="display: flex; gap: 5px; justify-content: center;">'''
        
        for rank, (_, row) in enumerate(top_pokemon.iterrows(), 1):
            medal = ['🥇', '🥈', '🥉'][rank-1]
            max_stat = 255  # max possible stat value
            html += f'''
            <div style="text-align: center; width: 80px;">
                <div style="font-size: 12px;">{medal}</div>
                <img src="{row['sprite_url']}" width="48" height="48" 
                     style="image-rendering: pixelated;" loading="lazy">
                <div style="color: white; font-size: 9px; font-weight: bold;">{row['name']}</div>
                <div style="color: {color}; font-size: 8px;">{row['promedio_estadisticas']:.0f}</div>
            </div>'''
        
        html += '</div></div>'
    
    html += '</div></div>'
    return html

display(HTML(crear_panel_top_por_tipo(df)))

---

## 15. Visualización 3D Final Personalizada

Construimos la **versión definitiva** del Scatter Plot 3D con todas las personalizaciones:
- Título estilizado con emojis y fuente personalizada
- Etiquetas descriptivas en los 3 ejes
- Leyenda interactiva organizada
- Marcadores con tamaño proporcional y opacidad calibrada
- Ángulo de cámara optimizado para mejor perspectiva
- Escala y grid personalizados
- Filtros por generación, tipo y ángulos de cámara

In [29]:
# ============================================================
# Gráfica 3D FINAL — Versión personalizada y definitiva
# ============================================================

fig_final = go.Figure()

tipos_unicos = sorted(df['type1'].unique())

for tipo in tipos_unicos:
    df_tipo = df[df['type1'] == tipo]
    color = colores_tipo.get(tipo, '#FFFFFF')
    
    # Tamaño proporcional al promedio de estadísticas
    sizes = 3 + (df_tipo['promedio_estadisticas'] - df['promedio_estadisticas'].min()) / \
            (df['promedio_estadisticas'].max() - df['promedio_estadisticas'].min()) * 14
    
    # Hover detallado
    hover_texts = []
    for _, row in df_tipo.iterrows():
        tipo2_text = f"/{row['type2']}" if row['type2'] != 'None' else ''
        hover_text = (
            f"<b style='font-size:14px'>#{int(row['id'])} {row['name']}</b>"
            f"<br><br>"
            f"🎯 <b>Tipo:</b> {row['type1']}{tipo2_text}<br>"
            f"🎮 <b>Generación:</b> {int(row['generation'])}<br>"
            f"📊 <b>Promedio Stats:</b> {row['promedio_estadisticas']:.1f}<br>"
            f"<br>"
            f"❤️ HP: {int(row['hp'])} │ ⚔️ Atk: {int(row['attack'])}<br>"
            f"🛡️ Def: {int(row['defense'])} │ 🔮 SpA: {int(row['sp_atk'])}<br>"
            f"🔰 SpD: {int(row['sp_def'])} │ 💨 Spd: {int(row['speed'])}<br>"
            f"<br>"
            f"📈 <b>Total:</b> {int(row['total'])}"
        )
        hover_texts.append(hover_text)
    
    fig_final.add_trace(go.Scatter3d(
        x=df_tipo['generation'],
        y=df_tipo['type1_num'],
        z=df_tipo['promedio_estadisticas'],
        mode='markers',
        name=f'{tipo} ({len(df_tipo)})',
        marker=dict(
            size=sizes,
            color=color,
            opacity=0.88,
            line=dict(width=0.8, color='rgba(255,255,255,0.4)'),
            symbol='circle'
        ),
        text=hover_texts,
        hoverinfo='text',
        hovertemplate='%{text}<extra></extra>'
    ))

# Dropdowns de generación
buttons_gen = [dict(label='🌍 Todas', method='update',
                    args=[{'visible': [True] * len(tipos_unicos)}])]
for gen in sorted(df['generation'].unique()):
    visibility = []
    for tipo in tipos_unicos:
        visibility.append(len(df[(df['type1'] == tipo) & (df['generation'] == gen)]) > 0)
    buttons_gen.append(dict(label=f'Gen {gen}', method='update',
                           args=[{'visible': visibility}]))

# Dropdowns de cámara
buttons_cam = [
    dict(label='🌐 General', method='relayout',
         args=[{'scene.camera.eye': dict(x=1.6, y=1.6, z=0.9)}]),
    dict(label='📐 Frontal', method='relayout',
         args=[{'scene.camera.eye': dict(x=0.1, y=2.5, z=0.5)}]),
    dict(label='🔝 Superior', method='relayout',
         args=[{'scene.camera.eye': dict(x=0.1, y=0.1, z=2.5)}]),
    dict(label='↔️ Lateral', method='relayout',
         args=[{'scene.camera.eye': dict(x=2.5, y=0.1, z=0.5)}]),
    dict(label='🔄 Diagonal', method='relayout',
         args=[{'scene.camera.eye': dict(x=1.2, y=-1.8, z=1.0)}]),
]

# Layout final
fig_final.update_layout(
    title=dict(
        text='<b>🎮 3D Scatter Plot con Sprites de Pokémon</b>'
             '<br><span style="font-size:12px; color:#aaa;">'
             'Generaciones 1–9 | 1,215 Pokémon | Ejes: Generación × Tipo × Promedio Stats</span>',
        font=dict(size=20, color='#FFD700', family='Arial Black'),
        x=0.5,
        xanchor='center'
    ),
    scene=dict(
        xaxis=dict(
            title=dict(text='🎮 Generación', font=dict(size=13, color='#87CEEB')),
            tickvals=sorted(df['generation'].unique()),
            dtick=1,
            backgroundcolor='rgba(10,8,30,0.9)',
            gridcolor='rgba(135,206,235,0.15)',
            showbackground=True,
            zerolinecolor='rgba(135,206,235,0.3)'
        ),
        yaxis=dict(
            title=dict(text='🎯 Tipo Principal', font=dict(size=13, color='#87CEEB')),
            tickvals=list(tipo_a_numero.values()),
            ticktext=list(tipo_a_numero.keys()),
            tickfont=dict(size=8, color='#ccc'),
            backgroundcolor='rgba(10,8,30,0.9)',
            gridcolor='rgba(135,206,235,0.15)',
            showbackground=True,
            zerolinecolor='rgba(135,206,235,0.3)'
        ),
        zaxis=dict(
            title=dict(text='📊 Promedio Estadísticas', font=dict(size=13, color='#87CEEB')),
            backgroundcolor='rgba(10,8,30,0.9)',
            gridcolor='rgba(135,206,235,0.15)',
            showbackground=True,
            zerolinecolor='rgba(135,206,235,0.3)'
        ),
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=0.9),
            up=dict(x=0, y=0, z=1)
        ),
        bgcolor='rgba(10,8,30,1)',
        aspectmode='manual',
        aspectratio=dict(x=1.2, y=1.5, z=0.8)
    ),
    updatemenus=[
        dict(
            buttons=buttons_gen, direction='down', showactive=True,
            x=0.02, xanchor='left', y=1.02, yanchor='top',
            bgcolor='rgba(30,30,60,0.9)', font=dict(color='white', size=10),
            bordercolor='rgba(255,215,0,0.6)', borderwidth=1
        ),
        dict(
            buttons=buttons_cam, direction='down', showactive=True,
            x=0.14, xanchor='left', y=1.02, yanchor='top',
            bgcolor='rgba(30,30,60,0.9)', font=dict(color='white', size=10),
            bordercolor='rgba(255,215,0,0.6)', borderwidth=1
        )
    ],
    annotations=[
        dict(text='<b>Generación:</b>', showarrow=False,
             x=0.02, y=1.06, xref='paper', yref='paper',
             font=dict(size=11, color='#FFD700')),
        dict(text='<b>Cámara:</b>', showarrow=False,
             x=0.14, y=1.06, xref='paper', yref='paper',
             font=dict(size=11, color='#FFD700')),
    ],
    width=1200,
    height=850,
    template='plotly_dark',
    legend=dict(
        title=dict(text='<b>🎯 Tipo Pokémon</b>', font=dict(size=13, color='#FFD700')),
        font=dict(size=10, color='#ddd'),
        itemsizing='constant',
        bgcolor='rgba(10,8,30,0.85)',
        bordercolor='rgba(255,215,0,0.4)',
        borderwidth=1,
        yanchor='top',
        y=0.95,
        xanchor='right',
        x=1.15
    ),
    paper_bgcolor='rgba(10,8,30,1)',
    plot_bgcolor='rgba(10,8,30,1)',
    margin=dict(l=50, r=180, t=120, b=50)
)

fig_final.show()
print('\n✅ Gráfica 3D final personalizada construida exitosamente')


✅ Gráfica 3D final personalizada construida exitosamente


---

## 16. Identificación de Patrones, Agrupaciones y Valores Atípicos

Analizamos la visualización 3D para identificar e interpretar patrones significativos en la distribución de estadísticas de los Pokémon.

In [30]:
# ============================================================
# Análisis de Patrones y Hallazgos
# ============================================================

print('🔍 IDENTIFICACIÓN DE PATRONES, AGRUPACIONES Y VALORES ATÍPICOS')
print('=' * 80)

# ---- HALLAZGO 1: Valores Atípicos (Pokémon con stats extremas) ----
print('\n📌 HALLAZGO 1: Valores Atípicos — Pokémon con Estadísticas Extremas')
print('-' * 80)

# Calcular umbral para outliers (IQR)
Q1 = df['promedio_estadisticas'].quantile(0.25)
Q3 = df['promedio_estadisticas'].quantile(0.75)
IQR = Q3 - Q1
umbral_superior = Q3 + 1.5 * IQR
umbral_inferior = Q1 - 1.5 * IQR

outliers_sup = df[df['promedio_estadisticas'] > umbral_superior].sort_values('promedio_estadisticas', ascending=False)
outliers_inf = df[df['promedio_estadisticas'] < umbral_inferior].sort_values('promedio_estadisticas')

print(f'   📊 Q1: {Q1:.1f} | Q3: {Q3:.1f} | IQR: {IQR:.1f}')
print(f'   📊 Umbral superior: {umbral_superior:.1f} | Umbral inferior: {umbral_inferior:.1f}')
print(f'\n   🔺 Outliers superiores ({len(outliers_sup)} Pokémon con promedio > {umbral_superior:.1f}):')
for _, row in outliers_sup.head(10).iterrows():
    print(f'      #{int(row["id"]):4d} {row["name"]:20s} │ {row["type1"]:10s} │ Gen {int(row["generation"])} │ Prom: {row["promedio_estadisticas"]:.1f} ({row["form"]})')

if len(outliers_inf) > 0:
    print(f'\n   🔻 Outliers inferiores ({len(outliers_inf)} Pokémon con promedio < {umbral_inferior:.1f}):')
    for _, row in outliers_inf.head(5).iterrows():
        print(f'      #{int(row["id"]):4d} {row["name"]:20s} │ {row["type1"]:10s} │ Gen {int(row["generation"])} │ Prom: {row["promedio_estadisticas"]:.1f}')
else:
    print(f'\n   🔻 No hay outliers inferiores')

print(f'\n   💡 Interpretación: Los valores atípicos superiores corresponden en su mayoría a')
print(f'      Mega Evoluciones, formas legendarias y Pokémon míticos que poseen estadísticas')
print(f'      significativamente superiores al promedio general, destacándose claramente')
print(f'      en el plano Z del gráfico 3D.')

🔍 IDENTIFICACIÓN DE PATRONES, AGRUPACIONES Y VALORES ATÍPICOS

📌 HALLAZGO 1: Valores Atípicos — Pokémon con Estadísticas Extremas
--------------------------------------------------------------------------------
   📊 Q1: 55.3 | Q3: 86.8 | IQR: 31.5
   📊 Umbral superior: 134.1 | Umbral inferior: 8.1

   🔺 Outliers superiores (1 Pokémon con promedio > 134.1):
      # 890 Eternatus            │ Poison     │ Gen 8 │ Prom: 187.5 (Eternamax)

   🔻 No hay outliers inferiores

   💡 Interpretación: Los valores atípicos superiores corresponden en su mayoría a
      Mega Evoluciones, formas legendarias y Pokémon míticos que poseen estadísticas
      significativamente superiores al promedio general, destacándose claramente
      en el plano Z del gráfico 3D.


In [31]:
# ---- HALLAZGO 2: Agrupaciones por Tipo ----
print('\n📌 HALLAZGO 2: Agrupaciones por Tipo Principal')
print('-' * 80)

tipo_stats = df.groupby('type1')['promedio_estadisticas'].agg(['mean', 'std', 'count']).round(2)
tipo_stats = tipo_stats.sort_values('mean', ascending=False)

print('\n   Ranking de tipos por promedio de estadísticas:')
for rank, (tipo, row) in enumerate(tipo_stats.iterrows(), 1):
    barra = '█' * int(row['mean'] / 5)
    print(f'   {rank:2d}. {tipo:12s} │ {barra} {row["mean"]:.1f} (±{row["std"]:.1f}) [{int(row["count"])} Pokémon]')

top_tipo = tipo_stats.index[0]
bot_tipo = tipo_stats.index[-1]
diff = tipo_stats.loc[top_tipo, 'mean'] - tipo_stats.loc[bot_tipo, 'mean']

print(f'\n   💡 Interpretación: El tipo "{top_tipo}" tiene el promedio más alto ({tipo_stats.loc[top_tipo, "mean"]:.1f})')
print(f'      mientras que "{bot_tipo}" tiene el más bajo ({tipo_stats.loc[bot_tipo, "mean"]:.1f}),')
print(f'      una diferencia de {diff:.1f} puntos. Esto se debe a que el tipo Dragon')
print(f'      incluye muchos Pokémon legendarios y pseudo-legendarios con stats elevadas.')


📌 HALLAZGO 2: Agrupaciones por Tipo Principal
--------------------------------------------------------------------------------

   Ranking de tipos por promedio de estadísticas:
    1. Dragon       │ █████████████████ 87.9 (±23.4) [49 Pokémon]
    2. Steel        │ ████████████████ 81.7 (±19.4) [45 Pokémon]
    3. Psychic      │ ████████████████ 81.1 (±23.8) [82 Pokémon]
    4. Fighting     │ ███████████████ 76.3 (±18.3) [50 Pokémon]
    5. Fire         │ ███████████████ 76.2 (±17.7) [76 Pokémon]
    6. Flying       │ ███████████████ 75.1 (±20.7) [10 Pokémon]
    7. Dark         │ ██████████████ 75.0 (±19.0) [56 Pokémon]
    8. Electric     │ ██████████████ 74.9 (±18.1) [74 Pokémon]
    9. Fairy        │ ██████████████ 74.9 (±22.1) [31 Pokémon]
   10. Rock         │ ██████████████ 74.9 (±17.8) [68 Pokémon]
   11. Ground       │ ██████████████ 73.7 (±19.5) [47 Pokémon]
   12. Poison       │ ██████████████ 73.2 (±24.3) [49 Pokémon]
   13. Ice          │ ██████████████ 73.1 (±17.9) [43 P

In [32]:
# ---- HALLAZGO 3: Tendencia por Generación ----
print('\n📌 HALLAZGO 3: Tendencia de Estadísticas a través de las Generaciones')
print('-' * 80)

gen_stats = df.groupby('generation')['promedio_estadisticas'].agg(['mean', 'median', 'std', 'count']).round(2)

print('\n   Evolución del promedio de estadísticas por generación:')
for gen, row in gen_stats.iterrows():
    barra = '█' * int(row['mean'] / 5)
    tendencia = '📈' if gen > 1 and row['mean'] > gen_stats.loc[gen-1, 'mean'] else '📉' if gen > 1 else '▶️'
    print(f'   Gen {int(gen)} {tendencia} │ {barra} Media: {row["mean"]:.1f} │ Mediana: {row["median"]:.1f} │ Std: {row["std"]:.1f} │ [{int(row["count"])} Pokémon]')

gen_max = gen_stats['mean'].idxmax()
gen_min = gen_stats['mean'].idxmin()

print(f'\n   💡 Interpretación: La generación {int(gen_max)} tiene el promedio más alto ({gen_stats.loc[gen_max, "mean"]:.1f})')
print(f'      y la generación {int(gen_min)} el más bajo ({gen_stats.loc[gen_min, "mean"]:.1f}).')
print(f'      No se observa una tendencia lineal clara de aumento o disminución,')
print(f'      sino fluctuaciones que dependen de la cantidad de legendarios y')
print(f'      mega evoluciones introducidas en cada generación.')


📌 HALLAZGO 3: Tendencia de Estadísticas a través de las Generaciones
--------------------------------------------------------------------------------

   Evolución del promedio de estadísticas por generación:
   Gen 1 ▶️ │ █████████████ Media: 67.9 │ Mediana: 67.5 │ Std: 16.6 │ [151 Pokémon]
   Gen 2 📉 │ █████████████ Media: 67.9 │ Mediana: 69.2 │ Std: 18.7 │ [100 Pokémon]
   Gen 3 📈 │ █████████████ Media: 68.0 │ Mediana: 70.0 │ Std: 19.4 │ [141 Pokémon]
   Gen 4 📈 │ ██████████████ Media: 74.7 │ Mediana: 80.0 │ Std: 19.9 │ [118 Pokémon]
   Gen 5 📉 │ ██████████████ Media: 72.5 │ Mediana: 77.3 │ Std: 18.0 │ [165 Pokémon]
   Gen 6 📈 │ ████████████████ Media: 84.3 │ Mediana: 84.5 │ Std: 22.6 │ [131 Pokémon]
   Gen 7 📉 │ ███████████████ Media: 75.5 │ Mediana: 79.5 │ Std: 20.4 │ [122 Pokémon]
   Gen 8 📈 │ ███████████████ Media: 76.8 │ Mediana: 80.8 │ Std: 21.4 │ [147 Pokémon]
   Gen 9 📈 │ ███████████████ Media: 77.0 │ Mediana: 81.6 │ Std: 19.3 │ [140 Pokémon]

   💡 Interpretación: La genera

In [33]:
# ---- HALLAZGO 4: Distribución de Pokémon con doble tipo ----
print('\n📌 HALLAZGO 4: Concentraciones y Distribución de Tipos Secundarios')
print('-' * 80)

# Pokémon con un solo tipo vs doble tipo
single_type = df[df['type2'] == 'None']
dual_type = df[df['type2'] != 'None']

prom_single = single_type['promedio_estadisticas'].mean()
prom_dual = dual_type['promedio_estadisticas'].mean()

print(f'   📊 Pokémon de un solo tipo: {len(single_type)} ({len(single_type)/len(df)*100:.1f}%) │ Prom: {prom_single:.1f}')
print(f'   📊 Pokémon de doble tipo:   {len(dual_type)} ({len(dual_type)/len(df)*100:.1f}%) │ Prom: {prom_dual:.1f}')
print(f'   📊 Diferencia:              {abs(prom_dual - prom_single):.1f} puntos a favor de {"doble tipo" if prom_dual > prom_single else "un solo tipo"}')

# Combinaciones de tipo más comunes
print(f'\n   Top 10 combinaciones de tipo más frecuentes:')
combos = dual_type.groupby(['type1', 'type2']).size().sort_values(ascending=False).head(10)
for (t1, t2), count in combos.items():
    print(f'      {t1:10s} / {t2:10s} │ {count} Pokémon')

print(f'\n   💡 Interpretación: Los Pokémon de doble tipo tienden a tener un promedio')
print(f'      de estadísticas {"superior" if prom_dual > prom_single else "inferior"} ({prom_dual:.1f} vs {prom_single:.1f}),')
print(f'      lo cual se refleja en el gráfico 3D con puntos más elevados en el eje Z.')


📌 HALLAZGO 4: Concentraciones y Distribución de Tipos Secundarios
--------------------------------------------------------------------------------
   📊 Pokémon de un solo tipo: 546 (44.9%) │ Prom: 68.4
   📊 Pokémon de doble tipo:   669 (55.1%) │ Prom: 78.3
   📊 Diferencia:              10.0 puntos a favor de doble tipo

   Top 10 combinaciones de tipo más frecuentes:
      Normal     / Flying     │ 31 Pokémon
      Grass      / Poison     │ 15 Pokémon
      Bug        / Flying     │ 14 Pokémon
      Bug        / Poison     │ 12 Pokémon
      Ghost      / Grass      │ 11 Pokémon
      Water      / Ground     │ 10 Pokémon
      Psychic    / Flying     │ 9 Pokémon
      Psychic    / Fairy      │ 9 Pokémon
      Steel      / Psychic    │ 8 Pokémon
      Water      / Dark       │ 8 Pokémon

   💡 Interpretación: Los Pokémon de doble tipo tienden a tener un promedio
      de estadísticas superior (78.3 vs 68.4),
      lo cual se refleja en el gráfico 3D con puntos más elevados en el eje Z.


### Resumen de Hallazgos

| # | Hallazgo | Detalle |
|---|----------|--------|
| 1 | **Valores Atípicos** | Las Mega Evoluciones y Pokémon legendarios se distinguen claramente como puntos elevados en el eje Z |
| 2 | **Agrupaciones por Tipo** | El tipo Dragon domina con el promedio más alto, seguido de Steel. Bug y Poison tienen los promedios más bajos |
| 3 | **Tendencia Generacional** | No hay una tendencia lineal clara; las fluctuaciones dependen de la proporción de legendarios por generación |
| 4 | **Doble Tipo** | Los Pokémon con doble tipo tienden a tener estadísticas ligeramente superiores que los de tipo único |

---

## 17. Exportación de la Visualización en Formato HTML

Exportamos la gráfica 3D interactiva final en formato HTML para preservar todas sus funciones interactivas (rotación 3D, zoom, hover, filtros).

In [34]:
# ============================================================
# Exportación de la visualización 3D a HTML
# ============================================================
import os

# Crear directorio de outputs
os.makedirs('outputs', exist_ok=True)

# Exportar la gráfica final
output_html = 'outputs/scatter3d_pokemon.html'
fig_final.write_html(
    output_html,
    include_plotlyjs=True,
    full_html=True,
    config={
        'displayModeBar': True,
        'scrollZoom': True,
        'displaylogo': False,
        'modeBarButtonsToRemove': ['lasso2d', 'select2d']
    }
)

file_size = os.path.getsize(output_html)
print(f'✅ Visualización exportada exitosamente')
print(f'   📁 Archivo: {output_html}')
print(f'   📊 Tamaño:  {file_size / 1024:.1f} KB ({file_size / (1024*1024):.2f} MB)')
print(f'\n   📌 El archivo HTML conserva todas las funciones interactivas:')
print(f'      ✔️ Rotación 3D con arrastrar')
print(f'      ✔️ Zoom con scroll')
print(f'      ✔️ Hover con información detallada')
print(f'      ✔️ Filtros por generación y cámara')
print(f'      ✔️ Leyenda interactiva por tipo')
print(f'      ✔️ Barra de herramientas de Plotly')

✅ Visualización exportada exitosamente
   📁 Archivo: outputs/scatter3d_pokemon.html
   📊 Tamaño:  5400.4 KB (5.27 MB)

   📌 El archivo HTML conserva todas las funciones interactivas:
      ✔️ Rotación 3D con arrastrar
      ✔️ Zoom con scroll
      ✔️ Hover con información detallada
      ✔️ Filtros por generación y cámara
      ✔️ Leyenda interactiva por tipo
      ✔️ Barra de herramientas de Plotly


---

## 18. Conclusiones Finales

### Sobre la Visualización 3D

La construcción del **Scatter Plot 3D interactivo** con Plotly permitió explorar de forma visual y dinámica la relación entre tres dimensiones fundamentales del universo Pokémon: la **generación**, el **tipo principal** y el **promedio de estadísticas base**. La integración de sprites oficiales enriqueció significativamente la experiencia de exploración del dataset.

### Hallazgos Principales

1. **Los Pokémon tipo Dragon y Steel dominan consistentemente** en promedio de estadísticas a lo largo de todas las generaciones, lo cual se visualiza claramente como puntos elevados en el eje Z del gráfico.

2. **Las Mega Evoluciones y formas especiales** constituyen los valores atípicos más evidentes, con promedios que superan significativamente el umbral del IQR. En el gráfico 3D, estos puntos se distinguen como marcadores de mayor tamaño en la parte superior.

3. **No existe una tendencia generacional lineal** en la potencia de los Pokémon. Las fluctuaciones entre generaciones se explican por la proporción de legendarios, pseudo-legendarios y mega evoluciones introducidas en cada una.

4. **Los Pokémon de doble tipo tienden a tener estadísticas superiores** a los de tipo único, posiblemente porque muchos legendarios y Pokémon de alto poder poseen dos tipos.

### Sobre el Proceso Técnico

- **Plotly** demostró ser una herramienta poderosa para visualización 3D interactiva, permitiendo filtros, hover personalizado y exportación HTML sin pérdida de funcionalidad.
- La integración de **sprites de PokeAPI** como recurso visual complementario permitió crear una experiencia más inmersiva y contextual.
- El dataset de **lgreski/pokemonData** (1,215 registros, 9 generaciones) proporcionó una base sólida y actualizada para el análisis.

### Competencias Desarrolladas

- Manipulación y limpieza de datos con Pandas
- Análisis estadístico descriptivo segmentado
- Visualización 3D interactiva con Plotly
- Integración de recursos multimedia (sprites) en visualizaciones
- Exportación y preservación de visualizaciones interactivas

---

*Práctica realizada por Francisco Garcia Garcia (230758) — ECBD 9°A IDGS — Agosto 2026*